# SigLIP2-SO400M Macro Router — Training
Garment macro classification: Upper/Lower/Underwear/Other

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
============================================================
   STANDALONE TRAINING — SigLIP2_SO400M
   Garment Macro Router (Upper/Lower/Underwear/Other)
============================================================
Fixes applied vs previous runs:
  1. LABEL_SMOOTHING = 0.0 (was 0.1 — caused 85%+ predictions below 0.60)
  2. Checkpoint verification after save (prevents silent corruption)
  3. Local-first save then copy to Drive (prevents Drive write crash)
"""

import os
import cv2
import numpy as np
from PIL import Image
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import math
import json
import time
import gc
import shutil

# ==============================================================
# CONFIG
# ==============================================================

MODEL_NAME = "SigLIP2_SO400M"
MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
BATCH_SIZE = 16
LR_HEAD = 2e-4
LR_BACKBONE = 5e-6
UNFREEZE_LAYERS = 6

TRAIN_ROOTS = [
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Training",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/random_data_sorted_4Jan_2026",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Merged_Data_Testing",
    "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data"
]

SAVE_DIR = "/content/drive/Shareddrives/Garment Type/Complete_dataset/Mar_data/models"
LOCAL_SAVE = "/content"  # Fast local SSD for checkpoint writes

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 60
EARLY_STOP_PATIENCE = 12
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.3
LABEL_SMOOTHING = 0.0      # FIX: was 0.1, caused low confidence in production
NUM_WORKERS = 4

APPLY_PREPROCESSING = True
BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

# ==============================================================
# CLASS MAPPING
# ==============================================================
UPPER_CLASSES = ["Blazer", "Jumpers", "Shirt", "T-shirt", "Dresses", "Fleece"]
LOWER_CLASSES = ["Jeans", "Trousers", "Jogging_Bottoms", "Skirts"]
OTHER_CLASSES = ["Towel"]
UNDERWEAR_CLASSES = ["Bra", "Knicker"]

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# ==============================================================
# PREPROCESSING
# ==============================================================

def preprocess_crop(img_np, bg_color=BACKGROUND_COLOR, margin_ratio=MARGIN_RATIO,
                    black_thresh=BLACK_THRESH):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > black_thresh, 255, 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)

    gray_bg = np.full_like(img_np, bg_color, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)

    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), bg_color, dtype=np.uint8)
        return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

    mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]

    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), bg_color, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    return Image.fromarray(cv2.cvtColor(square, cv2.COLOR_BGR2RGB))

# ==============================================================
# AUGMENTATIONS
# ==============================================================

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.3),
    transforms.RandomRotation(degrees=30, fill=128),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1),
                            scale=(0.85, 1.15), shear=10, fill=128),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3, fill=128),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15), value=0.5),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

normalize_fn = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

# ==============================================================
# DATASET
# ==============================================================

class RouterDataset(Dataset):
    def __init__(self, samples, transform, apply_preprocess=True):
        self.samples = samples
        self.transform = transform
        self.apply_preprocess = apply_preprocess

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        if self.apply_preprocess:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
            else:
                img = preprocess_crop(img_bgr)
        else:
            img = Image.open(path).convert("RGB")
        pixel_values = self.transform(img)
        pixel_values = normalize_fn(pixel_values)
        return pixel_values, label

# ==============================================================
# DATA COLLECTION
# ==============================================================

def map_class(cls_name):
    if cls_name in UPPER_CLASSES: return CLASS_TO_IDX["Upper"]
    if cls_name in LOWER_CLASSES: return CLASS_TO_IDX["Lower"]
    if cls_name in UNDERWEAR_CLASSES: return CLASS_TO_IDX["Underwear"]
    if cls_name in OTHER_CLASSES: return CLASS_TO_IDX["Other"]
    return None

def collect_samples(roots):
    samples = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for cls_name in sorted(os.listdir(root)):
            cls_path = os.path.join(root, cls_name)
            if not os.path.isdir(cls_path):
                continue
            label = map_class(cls_name)
            if label is None:
                continue
            for f in os.listdir(cls_path):
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    samples.append((os.path.join(cls_path, f), label))
    return samples

# ==============================================================
# MODEL
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=DROPOUT_RATE):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(x))

class CosineWarmupScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr_ratio=0.01):
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr_ratio = min_lr_ratio
        super().__init__(optimizer, last_epoch=-1)
    def get_lr(self):
        step = self.last_epoch
        if step < self.warmup_steps:
            scale = step / max(1, self.warmup_steps)
        else:
            progress = (step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            scale = self.min_lr_ratio + 0.5 * (1 - self.min_lr_ratio) * (1 + math.cos(math.pi * progress))
        return [base_lr * scale for base_lr in self.base_lrs]

def load_model():
    """Load SigLIP2 vision backbone."""
    try:
        from transformers import Siglip2VisionModel
        vision_model = Siglip2VisionModel.from_pretrained(MODEL_ID)
    except Exception:
        try:
            from transformers import SiglipVisionModel
            vision_model = SiglipVisionModel.from_pretrained(MODEL_ID)
        except Exception:
            from transformers import AutoModel
            full = AutoModel.from_pretrained(MODEL_ID)
            vision_model = full.vision_model
            del full

    hidden_size = vision_model.config.hidden_size

    # Freeze all, then unfreeze last N encoder layers + post_layernorm
    for p in vision_model.parameters():
        p.requires_grad = False
    for layer in vision_model.vision_model.encoder.layers[-UNFREEZE_LAYERS:]:
        for p in layer.parameters():
            p.requires_grad = True
    for p in vision_model.vision_model.post_layernorm.parameters():
        p.requires_grad = True

    return vision_model, hidden_size

def extract_features(model, pixel_values):
    return model(pixel_values=pixel_values).pooler_output


# ==============================================================
# PER-CLASS THRESHOLD COMPUTATION
# ==============================================================

def compute_class_thresholds(preds, gts, confs, target_precision=0.95):
    """
    For each class, find the minimum confidence threshold where
    precision >= target_precision. This gives per-class rejection
    thresholds for production use.

    Returns dict: {class_name: threshold}
    """
    thresholds = {}
    preds = np.array(preds)
    gts = np.array(gts)
    confs = np.array(confs)

    for cls_idx, cls_name in IDX_TO_CLASS.items():
        # All samples where model predicted this class
        pred_mask = preds == cls_idx
        if pred_mask.sum() == 0:
            thresholds[cls_name] = 0.99  # No predictions → very high threshold
            continue

        cls_confs = confs[pred_mask]
        cls_correct = (gts[pred_mask] == cls_idx)

        # Sort by confidence descending
        sort_idx = np.argsort(-cls_confs)
        cls_confs_sorted = cls_confs[sort_idx]
        cls_correct_sorted = cls_correct[sort_idx]

        # Walk from highest confidence down, find where precision drops below target
        cumulative_correct = np.cumsum(cls_correct_sorted)
        cumulative_total = np.arange(1, len(cls_correct_sorted) + 1)
        precisions = cumulative_correct / cumulative_total

        # Find the lowest confidence where precision >= target
        valid = np.where(precisions >= target_precision)[0]
        if len(valid) > 0:
            last_valid = valid[-1]
            thresholds[cls_name] = float(cls_confs_sorted[last_valid])
        else:
            # Can't reach target precision — use threshold where precision is maximized
            best_idx = np.argmax(precisions)
            thresholds[cls_name] = float(cls_confs_sorted[best_idx])

        # Also compute stats for logging
        correct_confs = cls_confs[cls_correct]
        wrong_confs = cls_confs[~cls_correct]

        n_total = int(pred_mask.sum())
        n_correct = int(cls_correct.sum())
        prec = n_correct / n_total if n_total > 0 else 0

        print(f"  {cls_name:12s}: threshold={thresholds[cls_name]:.3f}  "
              f"predictions={n_total}, correct={n_correct}, precision={prec:.3f}  "
              f"conf_correct={correct_confs.mean():.3f} conf_wrong={wrong_confs.mean():.3f}" if len(wrong_confs) > 0
              else f"  {cls_name:12s}: threshold={thresholds[cls_name]:.3f}  "
              f"predictions={n_total}, correct={n_correct}, precision={prec:.3f}  "
              f"conf_correct={correct_confs.mean():.3f} (no wrong predictions)")

    return thresholds


# ==============================================================
# CRASH-SAFE CHECKPOINTING (RESUMABLE)
# --------------------------------------------------------------
# - Writes "<MODEL>_last.pt" every epoch with FULL training state
# - Atomic: tmp file -> os.replace -> copy to Drive
# - Rotates 2 slots on Drive so a crash mid-copy never leaves you empty
# - On startup: auto-loads _last.pt and resumes from next epoch
# ==============================================================

def _atomic_torch_save(obj, final_path):
    """Write to <path>.tmp then atomically rename to <path>."""
    tmp_path = final_path + ".tmp"
    torch.save(obj, tmp_path)
    os.replace(tmp_path, final_path)

def save_last_checkpoint(state, filename, drive_dir):
    """Save resumable training state. Local -> verify -> Drive (rotating)."""
    local_path = os.path.join(LOCAL_SAVE, filename)
    _atomic_torch_save(state, local_path)

    # Verify locally
    try:
        v = torch.load(local_path, map_location="cpu", weights_only=False)
        assert v["epoch"] == state["epoch"]
        del v
    except Exception as e:
        print(f"  LOCAL LAST CKPT VERIFY FAILED: {e}")
        return False

    # Rotate on Drive: current _last.pt -> _last_prev.pt, then copy new
    drive_path = os.path.join(drive_dir, filename)
    drive_prev = os.path.join(drive_dir, filename.replace(".pt", "_prev.pt"))
    try:
        if os.path.exists(drive_path):
            # rotate: move old drive copy to _prev (best-effort)
            try:
                shutil.copy2(drive_path, drive_prev)
            except Exception:
                pass
        # copy new checkpoint to drive
        tmp_drive = drive_path + ".tmp"
        shutil.copy2(local_path, tmp_drive)
        os.replace(tmp_drive, drive_path)
    except Exception as e:
        print(f"  Drive copy of last-ckpt failed (local copy still safe): {e}")
        return False
    return True

def try_load_last_checkpoint(filename, drive_dir):
    """Look for a resumable checkpoint. Tries local, drive, drive_prev in order."""
    candidates = [
        os.path.join(LOCAL_SAVE, filename),
        os.path.join(drive_dir, filename),
        os.path.join(drive_dir, filename.replace(".pt", "_prev.pt")),
    ]
    for p in candidates:
        if not os.path.exists(p):
            continue
        try:
            ckpt = torch.load(p, map_location="cpu", weights_only=False)
            # sanity check
            _ = ckpt["epoch"]; _ = ckpt["vision_model"]; _ = ckpt["optimizer"]
            print(f"  Resume checkpoint found at: {p}")
            print(f"     -> resuming from epoch {ckpt['epoch']+1}, best_f1={ckpt.get('best_f1',0):.4f}")
            return ckpt
        except Exception as e:
            print(f"  Corrupt/incomplete ckpt at {p}: {e}")
            continue
    print("  No resume checkpoint found - starting fresh.")
    return None

# ==============================================================
# CHECKPOINT SAVE + VERIFY
# ==============================================================

def save_checkpoint(vision_model, classifier, hidden_size, best_f1, best_epoch,
                    filename, drive_path):
    """Save locally, verify, then copy to Drive."""
    local_path = os.path.join(LOCAL_SAVE, filename)
    ckpt = {
        "vision_model": vision_model.state_dict(),
        "classifier": classifier.state_dict(),
        "class_to_idx": CLASS_TO_IDX,
        "hidden_size": hidden_size,
        "best_f1": best_f1,
        "epoch": best_epoch,
        "model_name": MODEL_NAME,
        "label_smoothing": LABEL_SMOOTHING,
    }

    # Save locally
    torch.save(ckpt, local_path)

    # Verify: reload and check F1 matches
    verify = torch.load(local_path, map_location="cpu", weights_only=False)
    if abs(verify["best_f1"] - best_f1) > 1e-6:
        print(f"  ❌ CHECKPOINT VERIFICATION FAILED! Saved F1={verify['best_f1']}, expected={best_f1}")
        return False
    print(f"  ✅ Checkpoint verified: F1={verify['best_f1']:.4f}, epoch={verify['epoch']}")
    del verify

    # Copy to Drive
    try:
        shutil.copy2(local_path, drive_path)
        print(f"  ✅ Copied to Drive: {drive_path}")
    except Exception as e:
        print(f"  ⚠️ Drive copy failed (local copy still safe): {e}")

    return True

# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print(f"   TRAINING: {MODEL_NAME}")
    print(f"   Model: {MODEL_ID}")
    print(f"   Label smoothing: {LABEL_SMOOTHING} (fixed — was 0.1)")
    print("=" * 60)
    print(f"Device: {DEVICE}")

    os.makedirs(SAVE_DIR, exist_ok=True)

    # Collect data
    print("\nCollecting samples...")
    all_samples = collect_samples(TRAIN_ROOTS)
    print(f"Total: {len(all_samples)}")

    labels = [l for _, l in all_samples]
    train_samples, val_samples = train_test_split(
        all_samples, test_size=0.2, stratify=labels, random_state=42
    )
    print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

    # Class weights
    train_labels = [l for _, l in train_samples]
    train_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
    print("\nClass distribution:")
    for i in range(NUM_CLASSES):
        print(f"  {IDX_TO_CLASS[i]}: {train_counts[i]}")

    class_weights = torch.tensor(
        1.0 / (train_counts + 1e-6), dtype=torch.float32, device=DEVICE
    )
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES

    # Load model
    vision_model, hidden_size = load_model()
    vision_model = vision_model.to(DEVICE)

    trainable = sum(p.numel() for p in vision_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in vision_model.parameters())
    print(f"\nParams: {total:,}, Trainable: {trainable:,} ({100*trainable/total:.1f}%)")

    classifier = ClassifierHead(hidden_size, NUM_CLASSES).to(DEVICE)

    # Data loaders
    train_loader = DataLoader(
        RouterDataset(train_samples, train_transforms, APPLY_PREPROCESSING),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, drop_last=True,
    )
    val_loader = DataLoader(
        RouterDataset(val_samples, val_transforms, APPLY_PREPROCESSING),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True,
    )

    # Optimizer
    backbone_params = [p for p in vision_model.parameters() if p.requires_grad]
    head_params = list(classifier.parameters())
    optimizer = torch.optim.AdamW([
        {"params": head_params, "lr": LR_HEAD},
        {"params": backbone_params, "lr": LR_BACKBONE},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = len(train_loader) * 3
    scheduler = CosineWarmupScheduler(optimizer, warmup_steps, total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda")

    # Training
    best_f1 = 0.0
    best_epoch = 0
    patience = 0
    epoch_logs = []
    best_preds, best_gts, best_confs = [], [], []
    start_epoch = 0

    model_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_best.pt")
    last_ckpt_name = f"{MODEL_NAME}_last.pt"
    start_time = time.time()

    # --- RESUME IF POSSIBLE ---
    print("\n[resume] checking for existing training checkpoint...")
    resume = try_load_last_checkpoint(last_ckpt_name, SAVE_DIR)
    if resume is not None:
        vision_model.load_state_dict(resume["vision_model"])
        classifier.load_state_dict(resume["classifier"])
        optimizer.load_state_dict(resume["optimizer"])
        scheduler.load_state_dict(resume["scheduler"])
        scaler.load_state_dict(resume["scaler"])
        start_epoch     = resume["epoch"] + 1
        best_f1         = resume.get("best_f1", 0.0)
        best_epoch      = resume.get("best_epoch", 0)
        patience        = resume.get("patience", 0)
        epoch_logs      = resume.get("epoch_logs", [])
        best_preds      = resume.get("best_preds", [])
        best_gts        = resume.get("best_gts", [])
        best_confs      = resume.get("best_confs", [])
        # RNG state for reproducibility across resumes
        if "rng_torch" in resume:
            torch.set_rng_state(resume["rng_torch"])
        if "rng_cuda" in resume and torch.cuda.is_available():
            try: torch.cuda.set_rng_state_all(resume["rng_cuda"])
            except Exception: pass
        if "rng_numpy" in resume:
            np.random.set_state(resume["rng_numpy"])
        print(f"[resume] starting at epoch {start_epoch+1}/{EPOCHS} (best F1 so far: {best_f1:.4f})")
        del resume
        gc.collect(); torch.cuda.empty_cache()

    for epoch in range(start_epoch, EPOCHS):
        # Train
        vision_model.train()
        classifier.train()
        total_loss = 0.0

        train_bar = tqdm(train_loader, desc=f"  E{epoch+1}/{EPOCHS} [T]", leave=False)
        for px, y in train_bar:
            px = px.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                feats = extract_features(vision_model, px)
                logits = classifier(feats)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(backbone_params + head_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)

        # Validate
        vision_model.eval()
        classifier.eval()
        preds, gts, confs = [], [], []
        val_loss = 0.0

        with torch.no_grad(), torch.amp.autocast("cuda"):
            for px, y in val_loader:
                px = px.to(DEVICE, non_blocking=True)
                y_dev = y.to(DEVICE, non_blocking=True)
                feats = extract_features(vision_model, px)
                logits = classifier(feats)
                val_loss += criterion(logits, y_dev).item()
                probs = torch.softmax(logits, dim=1)
                max_confs, max_preds = probs.max(dim=1)
                preds.extend(max_preds.cpu().numpy())
                confs_batch = max_confs.cpu().numpy()
                confs.extend(confs_batch)
                gts.extend(y.numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(gts, preds, average="macro")

        epoch_logs.append({
            "epoch": epoch + 1,
            "train_loss": round(avg_loss, 5),
            "val_loss": round(avg_val_loss, 5),
            "val_f1": round(val_f1, 4),
        })

        print(f"  E{epoch+1:03d} | Loss: {avg_loss:.4f}/{avg_val_loss:.4f} | F1: {val_f1:.4f}", end="")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch + 1
            patience = 0
            best_preds = preds.copy()
            best_gts = gts.copy()
            best_confs = confs.copy()

            ok = save_checkpoint(vision_model, classifier, hidden_size,
                                 best_f1, best_epoch,
                                 f"{MODEL_NAME}_best.pt", model_path)
            print(f" >>> BEST {'✅' if ok else '❌'}")
        else:
            patience += 1
            print(f" ({patience}/{EARLY_STOP_PATIENCE})")
            if patience >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        # --- Save resumable "last" checkpoint every epoch ---
        last_state = {
            "epoch": epoch,
            "vision_model": vision_model.state_dict(),
            "classifier": classifier.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "patience": patience,
            "epoch_logs": epoch_logs,
            "best_preds": best_preds,
            "best_gts": best_gts,
            "best_confs": best_confs,
            "hidden_size": hidden_size,
            "class_to_idx": CLASS_TO_IDX,
            "model_name": MODEL_NAME,
            "label_smoothing": LABEL_SMOOTHING,
            "rng_torch": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "rng_numpy": np.random.get_state(),
        }
        save_last_checkpoint(last_state, last_ckpt_name, SAVE_DIR)
        del last_state

    elapsed = time.time() - start_time

    # Compute per-class thresholds
    class_thresholds = {}
    if best_preds and best_confs:
        print("\nComputing per-class confidence thresholds (target precision=95%):")
        class_thresholds = compute_class_thresholds(best_preds, best_gts, best_confs)
        print(f"\n  Thresholds: {class_thresholds}")

        # Re-save checkpoint with thresholds included
        print("\n  Re-saving checkpoint with per-class thresholds...")
        local_path = os.path.join(LOCAL_SAVE, f"{MODEL_NAME}_best.pt")
        ckpt = torch.load(local_path, map_location="cpu", weights_only=False)
        ckpt["class_thresholds"] = class_thresholds
        torch.save(ckpt, local_path)
        try:
            shutil.copy2(local_path, model_path)
            print(f"  ✅ Checkpoint updated with thresholds")
        except Exception as e:
            print(f"  ⚠️ Drive copy failed: {e}")

    # Final report
    if best_preds and best_gts:
        report = classification_report(best_gts, best_preds,
                                       target_names=list(CLASS_TO_IDX.keys()),
                                       output_dict=True)
        cm = confusion_matrix(best_gts, best_preds)
    else:
        report = {cls: {"f1-score": 0} for cls in CLASS_TO_IDX}
        cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

    print(f"\n{'='*60}")
    print(f"  {MODEL_NAME} DONE")
    print(f"  Best F1: {best_f1:.4f} @ epoch {best_epoch}")
    print(f"  Time: {elapsed/60:.1f} min")
    print(f"{'='*60}")
    print("\nPer-class F1:")
    for cls in CLASS_TO_IDX:
        print(f"  {cls:12s}: {report[cls]['f1-score']:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  {'':>12}", end="")
    for name in CLASS_TO_IDX: print(f"{name:>10}", end="")
    print()
    for i, name in enumerate(CLASS_TO_IDX):
        print(f"  {name:>12}", end="")
        for j in range(NUM_CLASSES): print(f"{cm[i][j]:>10}", end="")
        print()

    # Verify final checkpoint one more time
    print(f"\n🔍 Final checkpoint verification:")
    final_ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    print(f"  Checkpoint F1: {final_ckpt['best_f1']:.4f}, Epoch: {final_ckpt['epoch']}")
    assert abs(final_ckpt["best_f1"] - best_f1) < 1e-6, "CHECKPOINT MISMATCH!"
    print(f"  ✅ Verified — checkpoint is correct")

    # Save training log
    log_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_training_log.json")
    with open(log_path, "w") as f:
        json.dump({
            "model": MODEL_NAME,
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "total_epochs": epoch + 1,
            "train_time_min": round(elapsed / 60, 1),
            "label_smoothing": LABEL_SMOOTHING,
            "per_class_f1": {cls: round(report[cls]["f1-score"], 4) for cls in CLASS_TO_IDX},
            "confusion_matrix": cm.tolist(),
            "class_thresholds": class_thresholds,
            "epoch_logs": epoch_logs,
        }, f, indent=2)
    print(f"  Training log saved: {log_path}")

if __name__ == "__main__":
    main()